# TRAINING

In [1]:
# 1. NVIDIA-Treiber installieren (aktuellste Version, Download von NVIDIA.com)
# 2. CUDA Toolkit installieren (https://developer.nvidia.com/cuda-downloads)
# 3. Ultralytics installieren (automatisch inkl. torch/cuda support)
!python -m pip install --upgrade pip
%pip install ultralytics --upgrade
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.4 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.3.164
    Uninstalling ultralytics-8.3.164:
      Successfully uninstalled ultralytics-8.3.164
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.


Nach der Installation prüfe in Python, ob CUDA verfügbar ist:

In [2]:
import torch
print(torch.cuda.is_available())  # True = CUDA bereit


True


# Pfad zur YAML Datei eintragen

Passe bei Bedarf die Parameter an (epochs, batch, imgsz …)

Starte das Skript – Training läuft!

In [3]:
from ultralytics import YOLO

# --- Einstellungen (bitte ggf. anpassen) ---

MODEL_PATH = 'yolo11n.pt'               # Nano Modell, schnell und ressourcensparend (Standard)

DATASET_YAML = r"C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\04_splitted\data.yaml"  # Pfad zur dataset.yaml (wie vorhin erzeugt)


WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\hagmmart\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Trainings-Argumente einstellen

| Argument       | Bedeutung (einfach erklärt)                                                          | Default           |
| -------------- | ------------------------------------------------------------------------------------ | ----------------- |
| `epochs`       | Wie oft wird der ganze Datensatz "durchtrainiert"? (Mehr = besser, aber auch länger) | 100               |
| `imgsz`        | Bildgrösse, auf die vor Training skaliert wird.                                      | 640               |
| `batch`        | Wie viele Bilder werden gleichzeitig an die GPU/CPU geschickt?                       | 16                |
| `device`       | 0 = erste GPU, 1 = zweite GPU, 'cpu' = Prozessor                                     | 0                 |
| `optimizer`    | Optimierungsverfahren, das das Lernen steuert.                                       | 'auto'            |
| `lr0`          | Startwert der Lernrate                                                               | 0.01              |
| `lrf`          | Endwert der Lernrate, als Anteil von lr0                                             | 0.01              |
| `momentum`     | Schwung der Gewichtsaktualisierung (wichtig bei SGD)                                 | 0.937             |
| `weight_decay` | Bestraft zu komplexe Modelle (Überanpassungsschutz)                                  | 0.0005            |
| `patience`     | Wieviele Epochen abwarten, wenn keine Verbesserung mehr kommt?                       | 50                |
| `cos_lr`       | Nutze Cosine-Learningrate, oft für längeres, sanfteres Lernen                        | False             |
| `project`      | Ordner, in dem alles gespeichert wird                                                | 'runs/train'      |
| `name`         | Name des Experiments (wird im Projektordner gespeichert)                             | 'yolo11n/_custom' |
| `pretrained`   | Beginne mit vortrainiertem Modell (schneller & meist bessere Ergebnisse)             | True              |
| `resume`       | Falls Training abbricht, kannst du damit weitermachen                                | False             |
| `val`          | Nach jedem Durchgang das Modell testen/validieren                                    | True              |
| `workers`      | CPU-Prozesse fürs Datenladen (mehr ist schneller, je nach PC)                        | 4                 |


In [ ]:
# Trainingsparameter (alle wichtigsten Argumente, mit Defaults)
TRAIN_ARGS = {
    'epochs': 300,           # Wie oft das ganze Dataset gesehen wird (Standard: 100)
    'imgsz': 640,            # Bildgrösse, Standard ist 640 (je nach GPU bis 1280 möglich)
    'batch': 0.8,             # Batch-Grösse, Anzahl Bilder pro Schritt (je nach VRAM 8–64)
    'cache': False,           # Bilder im RAM cachen (schneller, aber mehr RAM nötig. WARNING cache='ram' may produce non-deterministic training results)
    'device': 0,             # GPU-Index (0=erste GPU, 'cpu'=CPU nutzen, [0,1] für mehrere GPUs)
    'optimizer': 'AdamW',     # Optimizer (z.B. SGD, Adam, AdamW oder 'auto' für Automatik)
    'lr0': 0.001,             # Start-Lernrate
    'lrf': 0.01,             # End-Lernrate als Faktor von lr0. Also 0.01 = 1% von lr0
    'momentum': 0.937,       # Nur für SGD, "Schwung" der Updates
    'weight_decay': 0.0005,  # Regularisierung gegen Überanpassung
    'patience': 25,          # Stoppt Training, falls Val-Loss nicht besser wird (Epochen)
    'cos_lr': True,         # Cosine-Learningrate statt linear (True/False)
    'project': r'C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\05_models', # Wohin die Ergebnisse gespeichert werden
    'name': '20250721 Training mit Fehlerbildern Produktion',# Name des Experiments (wird als Ordner erzeugt)
    'pretrained': True,      # Pretrained Modell nutzen (empfohlen)
    'resume': False,         # Training fortsetzen, falls gestoppt
    'val': True,             # Nach jedem Epoch validieren (empfohlen)
    'workers': 30,            # Anzahl CPU-Worker fürs Laden (je nach PC 2–30)
    'dropout': 0.20,          # Dropout-Wahrscheinlichkeit (0.0–0.5), 0=aus
    
    # Data Augmentation direkt im Training (Stärke, Methode)
    'hsv_h': 0.01,             # Farbtonverschiebung (0.0–0.5), 0=aus
    'hsv_s': 0.4,               # Sättigung (0.0–0.9)
    'hsv_v': 0.0,               # Helligkeit (0.0–0.9)
    'degrees': 0.01,             # Rotation in Grad, z. B. 10
    'translate': 0.1,           # Verschiebung als Anteil, z. B. 0.1 = 10%
    'scale': 0.1,               # Skalierung (0=aus, 0.5=±50%)
    'shear': 0.0,               # Scherung (0–2.0)
    'perspective': 0.1,         # Perspektivische Verzerrung (0–0.001)
    'flipud': 0.0,              # Vertikal spiegeln (Wahrscheinlichkeit, 0–1)
    'fliplr': 0.1,              # Horizontal spiegeln (Wahrscheinlichkeit, 0–1)
    'mosaic': 1.0,              # Mosaic-Augmentation (0–1), meist anlassen
    'mixup': 1,                 # Mixup (0–1), Bilder kombinieren

    # Loss-Anteile für Boxen, Klassifikation, Objekt
    'box': 7.5,                 # Box-Loss-Gewichtung (0–10), Standard: 7.5
    'cls': 0.5,                 # Klassifikations-Loss-Gewichtung. Empfohlen: 0.5
    'dfl': 1.5,                 # Distribution Focal Loss (0–10). Empfohlen: 1.5

    # Save/checkpoints
    'save_period': 50,          # Alle x Epochen ein Modell speichern (Default=50)
    'exist_ok': True,           # Vorhandene Ergebnisse überschreiben (False=Fehler)
    'verbose': True,            # Detailierte Ausgaben (False=weniger Text)

    # Advanced: Training nur auf bestimmten Klassen (nur falls gebraucht)
    # 'classes': [0, 1, 2, 3],        # Nur Klasse 0 und 2 trainieren

    # Seed für Reproduzierbarkeit
    'seed': 42                 # Zufallszahlgenerator initialisiere
}

---
### Training startet mit Ausführung des nächsten Skriptes

In [6]:
# ---- Schritt 1: Modell laden ----
model = YOLO(MODEL_PATH)

# ---- Schritt 2: Training starten ----
results = model.train(
    data=DATASET_YAML,
    epochs=TRAIN_ARGS['epochs'],
    imgsz=TRAIN_ARGS['imgsz'],
    batch=TRAIN_ARGS['batch'],
    cache=TRAIN_ARGS['cache'],
    device=TRAIN_ARGS['device'],
    optimizer=TRAIN_ARGS['optimizer'],
    lr0=TRAIN_ARGS['lr0'],
    lrf=TRAIN_ARGS['lrf'],
    momentum=TRAIN_ARGS['momentum'],
    weight_decay=TRAIN_ARGS['weight_decay'],
    patience=TRAIN_ARGS['patience'],
    cos_lr=TRAIN_ARGS['cos_lr'],
    project=TRAIN_ARGS['project'],
    name=TRAIN_ARGS['name'],
    pretrained=TRAIN_ARGS['pretrained'],
    resume=TRAIN_ARGS['resume'],
    val=TRAIN_ARGS['val'],
    workers=TRAIN_ARGS['workers'],
    dropout=TRAIN_ARGS['dropout'],
    hsv_h=TRAIN_ARGS['hsv_h'],
    hsv_s=TRAIN_ARGS['hsv_s'],
    hsv_v=TRAIN_ARGS['hsv_v'],
    degrees=TRAIN_ARGS['degrees'],
    translate=TRAIN_ARGS['translate'],
    scale=TRAIN_ARGS['scale'],
    shear=TRAIN_ARGS['shear'],
    perspective=TRAIN_ARGS['perspective'],
    flipud=TRAIN_ARGS['flipud'],
    fliplr=TRAIN_ARGS['fliplr'],
    mosaic=TRAIN_ARGS['mosaic'],
    mixup=TRAIN_ARGS['mixup'],
    box=TRAIN_ARGS['box'],
    cls=TRAIN_ARGS['cls'],
    dfl=TRAIN_ARGS['dfl'],
    save_period=TRAIN_ARGS['save_period'],
    exist_ok=TRAIN_ARGS['exist_ok'],
    verbose=TRAIN_ARGS['verbose'],
    # classes=TRAIN_ARGS['classes'],  # Nur wenn nötig
    seed=TRAIN_ARGS['seed']
)

# Das Training wird jetzt gestartet. Je nach Hardware kann das einige Zeit dauern.
# Die Ergebnisse werden im angegebenen Projekt-Ordner gespeichert.

Ultralytics 8.3.168  Python-3.13.3 torch-2.8.0.dev20250324+cu128 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=0.8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\04_splitted\data.yaml, degrees=0.01, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.1, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.4, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20250721 Training mit Fehlerbildern Produktion, nbs=64, nms=False, opset=No

train: Scanning C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\04_splitted\train\labels... 1341 images, 95 backgrounds, 0 corrupt: 100%|██████████| 1341/1341 [00:00<00:00, 3765.17it/s]

train: New cache created: C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\04_splitted\train\labels.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 80.0% CUDA memory utilization.
AutoBatch: CUDA:0 (NVIDIA RTX A4000) 15.99G total, 0.10G reserved, 0.06G allocated, 15.83G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
     2590620       6.444         0.457         34.42         321.8        (1, 3, 640, 640)                    list
     2590620       12.89         0.623         18.94         97.82        (2, 3, 640, 640)                    list
     2590620       25.78         0.931         26.32         91.99        (4, 3, 640, 640)                    list
     2590620       51.55         1.592         27.93         98.49        (8, 3, 640, 640)                    list
     2590620       103.1   

train: Scanning C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\04_splitted\train\labels.cache... 1341 images, 95 backgrounds, 0 corrupt: 100%|██████████| 1341/1341 [00:00<?, ?it/s]


...

    117/300      10.7G      1.569      2.157      1.569        668        640: 100%|██████████| 17/17 [00:08<00:00,  2.09it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.04it/s]

                   all        383       2883      0.976      0.478      0.518      0.289
EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 92, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



117 epochs completed in 0.345 hours.
Optimizer stripped from C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\05_models\20250721 Training mit Fehlerbildern Produktion\weights\last.pt, 5.5MB
Optimizer stripped from C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\05_models\20250721 Training mit Fehlerbildern Produktion\weights\best.pt, 5.5MB

Validating C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\05_models\20250721 Training mit Fehlerbildern Produktion\weights\best.pt...
Ultralytics 8.3.168  Python-3.13.3 torch-2.8.0.dev20250324+cu128 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.35it/s]


                   all        383       2883      0.951      0.456      0.508      0.303
               GEWINDE        351       1321      0.893      0.896      0.945      0.613
           DEFORMATION         88         93          1          0      0.111     0.0668
                 BODEN        351       1409      0.913      0.928      0.973      0.531
                PUNKTE         22         60          1          0    0.00406   0.000406
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to C:\Users\hagmmart\AI_Vision_Projects\M inner Tube  (B) 111.05387.000_V2\05_models\20250721 Training mit Fehlerbildern Produktion
